<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

# Mathematical Foundations. Automatic differentiation with PyTorch. Exercise

We define the following function:

$$ f(x,y)=\begin{cases} \displaystyle \sum_{k=1}^{n} \big(x^k \cdot y\big), & \text{if } y>0, \\[1.2em]
\log\!\big(1+x^2+y^2\big), & \text{if } y \leq 0, \end{cases}$$

with $\lfloor |y| \rfloor + 1$ (n is the integer part of the absolute value of $y$ +1), and $x,y\in \mathbb{R}$.

Tasks:
1. Implement the function in PyTorch.
2. Evaluate $f(x,y)$ and compute gradients for these cases:
    * $(x=1.0, y=2.0)$
    * $(x=1.0, y=-1.0)$
    * $(x=2.0, y=0.0)$
3. Inspect the computation graph (`grad_fn`and `next_functions`) for each case
4. Sketch the structure of the graph for each case
5. Discuss why the graph changes depending on the input.

In [1]:
import torch


def f(x, y):
    n = int(torch.floor(torch.abs(y)).item()) + 1
    if y > 0:
        c =  sum(x**k * y for k in range(1, n + 1))
    else:
        c = torch.log(1 + x**2 + y**2)
    return c

In [2]:
def evaluate_function(x_value, y_value):
    x = torch.tensor(float(x_value), requires_grad=True)
    y = torch.tensor(float(y_value), requires_grad=True)

    result = f(x, y)

    result.backward()

    print(f"x = {x_value}, y = {y_value}")
    print(f"f(x, y) = {result.item()}")
    print(f"df/dx = {x.grad.item()}")
    print(f"df/dy = {y.grad.item()}")
    print()

In [3]:
evaluate_function(1.0, 2.0)
evaluate_function(1.0, -1.0)
evaluate_function(2.0, 0.0)

x = 1.0, y = 2.0
f(x, y) = 6.0
df/dx = 12.0
df/dy = 3.0

x = 1.0, y = -1.0
f(x, y) = 1.0986123085021973
df/dx = 0.6666666865348816
df/dy = -0.6666666865348816

x = 2.0, y = 0.0
f(x, y) = 1.6094379425048828
df/dx = 0.800000011920929
df/dy = 0.0



/home/ruthaltmno/FoundationsMachineLearning/Practices/.venv/lib/python3.14/site-packages/torch/autograd/graph.py:1072: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12050). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [4]:
def inspect_graph(x_value, y_value):
    x = torch.tensor(float(x_value), requires_grad=True)
    y = torch.tensor(float(y_value), requires_grad=True)

    result = f(x, y)

    print(f"\n(x, y) = ({x_value}, {y_value})")
    print(f"result = {result}")
    print(f"grad_fn = {result.grad_fn}")
    print("next_functions:")

    for function in result.grad_fn.next_functions:
        print("   ", function)

In [5]:
inspect_graph(1.0, 2.0)
inspect_graph(1.0, -1.0)
inspect_graph(2.0, 0.0)


(x, y) = (1.0, 2.0)
result = 6.0
grad_fn = <AddBackward0 object at 0x7f621362bd90>
next_functions:
    (<AddBackward0 object at 0x7f62135bdb70>, 0)
    (<MulBackward0 object at 0x7f62135bee30>, 0)

(x, y) = (1.0, -1.0)
result = 1.0986123085021973
grad_fn = <LogBackward0 object at 0x7f62135bcfd0>
next_functions:
    (<AddBackward0 object at 0x7f62135bee30>, 0)

(x, y) = (2.0, 0.0)
result = 1.6094379425048828
grad_fn = <LogBackward0 object at 0x7f621362bd90>
next_functions:
    (<AddBackward0 object at 0x7f621495a8f0>, 0)
